In [1]:
# %% [markdown]
# # O*NET Task Time Estimation by Occupation
#
# Purpose:
# Estimate how many hours per week a typical worker in each occupation
# spends on each task, using Gemini via Vertex AI.
#
# Output:
# One normalized task-time file with:
# occ_code, occ_title, task_id, task_title, raw_task_weekly_time, normalized_time
#
# Resume-safe:
# Intermediate occupation-level checkpoint saved to parquet after every N occupations.

In [2]:
# %%
import json
import time
import logging
import pathlib
import threading
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm.notebook import tqdm

from google import genai
from google.genai import types

# ── Config ────────────────────────────────────────────────────────────
PROJECT_ID = "eloundou-new-scores"
LOCATION   = "global"
MODEL_NAME = "gemini-3.1-pro-preview"

BASE = pathlib.Path("/Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500")

INPUT_FILE = BASE / "Eloundou_New" / "full_labelset_new.tsv"

INTERMEDIATE_FILE = BASE / "Time" / "_checkpoint_task_time_estimation.parquet"
OUTPUT_FILE       = BASE / "Time" / "task_time_estimates_normalized.csv"

FRESH_START = False
DRY_RUN = False

MAX_OCCUPATIONS = None
PARALLEL_OCCUPATION_WORKERS = 2
SAVE_EVERY_N_OCCUPATIONS = 10

TEMPERATURE = 0.1
TOP_P = 0.95
MAX_OUTPUT_TOKENS = 20000

MAX_RETRIES = 5
INITIAL_BACKOFF = 2.0

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("task_time")
log.info("Configuration loaded.")

# ── Client ───────────────────────────────────────────────────────────
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
log.info(f"GenAI client ready  project={PROJECT_ID}  location={LOCATION}  model={MODEL_NAME}")

14:47:10  INFO      Configuration loaded.
14:47:10  INFO      GenAI client ready  project=eloundou-new-scores  location=global  model=gemini-3.1-pro-preview


In [3]:
# %%
TIME_ESTIMATION_SYSTEM_PROMPT = """You are an expert labour-market analyst.

You estimate how much time workers in an occupation spend on each of their job tasks.

You will be given:
1. an occupation title
2. a list of task IDs and task statements for that occupation

Your job:
Estimate how many hours per week a typical worker in that occupation spends on each task.

Important rules:
1. Estimate each task independently based on what seems realistic.
2. Do not force the hours to sum to exactly 40 or any other total.
3. We will normalize the results afterward.
4. Return ONLY valid JSON.
5. The JSON must map each task_id to a numeric value representing estimated hours per week.
6. No markdown, no commentary, no explanation.

Example format:
{
  "1234": 2.5,
  "1235": 6.0,
  "1236": 0.75
}"""


def build_occupation_prompt(occupation_title: str, tasks: list[dict]) -> str:
    task_lines = []
    for t in tasks:
        task_lines.append(f"{t['task_id']}: {t['task_title']}")

    tasks_block = "\n".join(task_lines)

    return f"""You are estimating how much time workers in the occupation
“{occupation_title}” spend on each of their job tasks.
Below is the complete list of tasks for this occupation. For each
task, estimate how many hours per week a typical worker spends on
it.
Important: Don’t worry about making the hours sum to exactly 40 or
any specific total - we’ll normalize the results afterward. Just
give your best estimate for each task independently based on what
seems realistic.
Tasks:
{tasks_block}
Return ONLY a JSON object mapping each task_id to your estimated
hours per week, with no additional text, explanations, or
commentary. Format:
{{
 “task_id_1”: hours,
 “task_id_2”: hours,
 ...
}}"""

In [4]:
# %%
_token_log = {"prompt_tokens": 0, "completion_tokens": 0, "calls": 0}

_checkpoint_lock = threading.Lock()

GLOBAL_COOLDOWN = 0.0
COOLDOWN_DECAY  = 0.95
COOLDOWN_BUMP   = 2.0
COOLDOWN_MAX    = 60.0


def _sleep_before_call():
    global GLOBAL_COOLDOWN
    if GLOBAL_COOLDOWN > 0:
        time.sleep(GLOBAL_COOLDOWN)


def _on_success():
    global GLOBAL_COOLDOWN
    GLOBAL_COOLDOWN *= COOLDOWN_DECAY


def _on_429(wait_time: float | None = None):
    global GLOBAL_COOLDOWN
    bump = wait_time if wait_time is not None else COOLDOWN_BUMP
    GLOBAL_COOLDOWN = min(COOLDOWN_MAX, GLOBAL_COOLDOWN + bump)


def _estimate_tokens(text: str) -> int:
    return max(1, len(text) // 4)


def _call_time_model(user_prompt: str) -> str:
    last_exception = None

    for attempt in range(MAX_RETRIES):
        try:
            _sleep_before_call()

            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=TIME_ESTIMATION_SYSTEM_PROMPT + "\n\n" + user_prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    top_p=TOP_P,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    response_mime_type="application/json",
                ),
            )

            _on_success()

            raw = (response.text or "").strip()

            if hasattr(response, "usage_metadata") and response.usage_metadata:
                usage = response.usage_metadata
                _token_log["prompt_tokens"] += usage.prompt_token_count or 0
                _token_log["completion_tokens"] += usage.candidates_token_count or 0
            else:
                _token_log["prompt_tokens"] += _estimate_tokens(user_prompt)
                _token_log["completion_tokens"] += _estimate_tokens(raw)
            _token_log["calls"] += 1

            return raw

        except Exception as e:
            last_exception = e
            error_str = str(e)

            if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                wait_time = INITIAL_BACKOFF * (2 ** attempt)
                _on_429(wait_time)
                log.warning(
                    f"Rate limit hit (attempt {attempt + 1}/{MAX_RETRIES}), waiting {wait_time:.1f}s..."
                )
                time.sleep(wait_time)
                continue

            raise

    raise last_exception


def _repair_json_dict(raw: str) -> dict | None:
    try:
        return json.loads(raw)
    except Exception:
        pass

    start = raw.find("{")
    end = raw.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = raw[start:end + 1]
        try:
            return json.loads(candidate)
        except Exception:
            pass

    return None


def query_task_times_for_occupation(occ_code: str, occ_title: str, tasks: list[dict]) -> list[dict]:
    if DRY_RUN:
        return [
            {
                "occ_code": occ_code,
                "occ_title": occ_title,
                "task_id": t["task_id"],
                "task_title": t["task_title"],
                "raw_task_weekly_time": 1.0,
                "model_name": MODEL_NAME,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            for t in tasks
        ]

    prompt = build_occupation_prompt(occ_title, tasks)
    raw = _call_time_model(prompt)
    parsed = _repair_json_dict(raw)

    if parsed is None or not isinstance(parsed, dict):
        raise ValueError(f"Could not parse JSON response for occupation {occ_code}: {raw[:500]}")

    rows = []
    for t in tasks:
        val = parsed.get(str(t["task_id"]))
        if val is None:
            val = parsed.get(t["task_id"])

        try:
            val = float(val)
        except Exception:
            val = None

        rows.append({
            "occ_code": occ_code,
            "occ_title": occ_title,
            "task_id": t["task_id"],
            "task_title": t["task_title"],
            "raw_task_weekly_time": val,
            "model_name": MODEL_NAME,
            "timestamp": datetime.now(timezone.utc).isoformat(),
        })

    return rows

In [5]:
# %%
df_tasks = pd.read_csv(INPUT_FILE, sep="\t", low_memory=False)
log.info(f"Loaded {len(df_tasks):,} rows from {INPUT_FILE}")

required_cols = ["Task ID", "Task", "Title"]
missing = [c for c in required_cols if c not in df_tasks.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

occ_code_col_candidates = ["O*NET-SOC Code", "O*NET-SOC code", "occ_code", "soc_code"]
occ_code_col = None
for c in occ_code_col_candidates:
    if c in df_tasks.columns:
        occ_code_col = c
        break

if occ_code_col is None:
    raise KeyError(f"Could not find occupation code column. Available columns: {list(df_tasks.columns)}")

df_tasks["Task ID"] = df_tasks["Task ID"].astype(str).str.strip()
df_tasks["Task"] = df_tasks["Task"].astype(str).str.strip()
df_tasks["Title"] = df_tasks["Title"].astype(str).str.strip()
df_tasks[occ_code_col] = df_tasks[occ_code_col].astype(str).str.strip()

print("Using occupation code column:", occ_code_col)
df_tasks[[occ_code_col, "Title", "Task ID", "Task"]].head(3)

14:47:10  INFO      Loaded 19,265 rows from /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Eloundou_New/full_labelset_new.tsv


Using occupation code column: O*NET-SOC Code


,O*NET-SOC Code,Title,Task ID,Task
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...


In [6]:
# %%
# Build occupation payloads
grouped = df_tasks.groupby([occ_code_col, "Title"])
occupation_payloads = []

for (occ_code, occ_title), grp in grouped:
    tasks = [
        {"task_id": str(row["Task ID"]), "task_title": str(row["Task"])}
        for _, row in grp.iterrows()
    ]
    occupation_payloads.append({
        "occ_code": str(occ_code),
        "occ_title": str(occ_title),
        "tasks": tasks,
    })

if MAX_OCCUPATIONS is not None:
    occupation_payloads = occupation_payloads[:MAX_OCCUPATIONS]
    log.info(f"MAX_OCCUPATIONS limit applied: processing only {len(occupation_payloads):,} occupations")

log.info(f"Built payloads for {len(occupation_payloads):,} occupations")


14:47:10  INFO      Built payloads for 923 occupations


In [7]:
# %%
if FRESH_START and INTERMEDIATE_FILE.exists():
    INTERMEDIATE_FILE.unlink()
    log.info("FRESH_START=True → deleted old checkpoint")

already_done = set()

if INTERMEDIATE_FILE.exists():
    df_ckpt = pd.read_parquet(INTERMEDIATE_FILE)
    done_occ = df_ckpt.groupby("occ_code")["raw_task_weekly_time"].apply(lambda s: s.notna().all())
    already_done = set(done_occ[done_occ].index.astype(str))
    log.info(f"Resuming: {len(already_done):,} occupations already done")
else:
    log.info("No checkpoint found — starting fresh")


def _save_checkpoint(rows: list[dict]) -> None:
    new_df = pd.DataFrame(rows)
    with _checkpoint_lock:
        if INTERMEDIATE_FILE.exists():
            old = pd.read_parquet(INTERMEDIATE_FILE)
            combined = pd.concat([old, new_df], ignore_index=True)
        else:
            combined = new_df.copy()

        combined = combined.sort_values(["occ_code", "task_id", "timestamp"])
        combined = combined.drop_duplicates(subset=["occ_code", "task_id"], keep="last")
        combined.to_parquet(INTERMEDIATE_FILE, index=False)

14:47:10  INFO      No checkpoint found — starting fresh


In [8]:
# %%
def _run_one_occupation(payload: dict) -> list[dict]:
    occ_code = str(payload["occ_code"])
    occ_title = str(payload["occ_title"])
    tasks = payload["tasks"]

    try:
        return query_task_times_for_occupation(occ_code, occ_title, tasks)
    except Exception as exc:
        log.error(f"Occupation failed {occ_code} | {occ_title}: {exc}")
        return [
            {
                "occ_code": occ_code,
                "occ_title": occ_title,
                "task_id": t["task_id"],
                "task_title": t["task_title"],
                "raw_task_weekly_time": None,
                "model_name": MODEL_NAME,
                "timestamp": datetime.now(timezone.utc).isoformat(),
            }
            for t in tasks
        ]


def estimate_all_occupations(payloads: list[dict], already_done: set[str]) -> list[dict]:
    pending = [p for p in payloads if str(p["occ_code"]) not in already_done]
    log.info(f"Occupations to process: {len(pending):,} | already done: {len(already_done):,}")

    results = []
    completed_occ = 0

    with ThreadPoolExecutor(max_workers=PARALLEL_OCCUPATION_WORKERS) as pool:
        futures = {pool.submit(_run_one_occupation, p): p for p in pending}

        with tqdm(total=len(futures), desc="Estimating occupations") as pbar:
            for future in as_completed(futures):
                rows = future.result()
                results.extend(rows)
                completed_occ += 1
                pbar.update(1)

                if completed_occ % SAVE_EVERY_N_OCCUPATIONS == 0:
                    _save_checkpoint(results)
                    log.info(f"Checkpoint saved after {completed_occ:,} occupations")

    if results:
        _save_checkpoint(results)
        log.info("Final checkpoint saved")

    return results

In [9]:
# %%
results = estimate_all_occupations(occupation_payloads, already_done)
log.info(f"Finished current run. Rows produced this run: {len(results):,}")

14:47:10  INFO      Occupations to process: 923 | already done: 0
14:47:10  INFO      AFC is enabled with max remote calls: 10.
14:47:10  INFO      AFC is enabled with max remote calls: 10.


Estimating occupations:   0%|          | 0/923 [00:00<?, ?it/s]

14:47:24  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
14:47:24  INFO      AFC is enabled with max remote calls: 10.
14:47:35  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
14:47:35  INFO      AFC is enabled with max remote calls: 10.
14:47:36  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateContent "HTTP/1.1 200 OK"
14:47:36  INFO      AFC is enabled with max remote calls: 10.
14:47:49  INFO      HTTP Request: POST https://aiplatform.googleapis.com/v1beta1/projects/eloundou-new-scores/locations/global/publishers/google/models/gemini-3.1-pro-preview:generateCont

In [10]:
# %%
df_est = pd.read_parquet(INTERMEDIATE_FILE)
log.info(f"Checkpoint rows loaded: {len(df_est):,}")

df_est["raw_task_weekly_time"] = pd.to_numeric(df_est["raw_task_weekly_time"], errors="coerce")

# Keep only rows with valid estimated times
df_valid = df_est.dropna(subset=["raw_task_weekly_time"]).copy()

# Merge back to make sure occupation-task universe matches source file
df_base = (
    df_tasks[[occ_code_col, "Title", "Task ID", "Task"]]
    .drop_duplicates()
    .rename(columns={
        occ_code_col: "occ_code",
        "Title": "occ_title",
        "Task ID": "task_id",
        "Task": "task_title",
    })
)

df_out = df_base.merge(
    df_valid[["occ_code", "occ_title", "task_id", "raw_task_weekly_time"]],
    on=["occ_code", "occ_title", "task_id"],
    how="left",
)

# Normalize within occupation
occ_sum = df_out.groupby("occ_code")["raw_task_weekly_time"].transform("sum")
df_out["normalized_time"] = df_out["raw_task_weekly_time"] / occ_sum

# Optional safety: if an occupation sum is 0 or missing, normalized_time stays NaN
df_out.loc[occ_sum <= 0, "normalized_time"] = pd.NA

# Final column order
df_out = df_out[
    ["occ_code", "occ_title", "task_id", "task_title", "raw_task_weekly_time", "normalized_time"]
].sort_values(["occ_code", "task_id"])

df_out.to_csv(OUTPUT_FILE, index=False)
log.info(f"Saved final output: {OUTPUT_FILE}")

print(df_out.head())
print("\nRows:", len(df_out))
print("Occupations:", df_out["occ_code"].nunique())
print("Tasks with raw estimate:", df_out["raw_task_weekly_time"].notna().sum())
print("Tasks with normalized estimate:", df_out["normalized_time"].notna().sum())

17:37:24  INFO      Checkpoint rows loaded: 19,265
17:37:24  INFO      Saved final output: /Users/nicobagnoli/Documents/PYTHON/Standard-Analysis-Nico/SMP500/Time/task_time_estimates_normalized.csv


      occ_code         occ_title task_id  \
16  11-1011.00  Chief Executives   20461   
0   11-1011.00  Chief Executives    8823   
5   11-1011.00  Chief Executives    8824   
2   11-1011.00  Chief Executives    8825   
3   11-1011.00  Chief Executives    8826   

                                           task_title  raw_task_weekly_time  \
16  Review and analyze legislation, laws, or publi...                   1.0   
0   Direct or coordinate an organization's financi...                   4.0   
5   Confer with board members, organization offici...                   8.0   
2   Analyze operations to evaluate performance of ...                   5.0   
3   Direct, plan, or implement policies, objective...                   6.0   

    normalized_time  
16         0.013812  
0          0.055249  
5          0.110497  
2          0.069061  
3          0.082873  

Rows: 19265
Occupations: 923
Tasks with raw estimate: 19265
Tasks with normalized estimate: 19265


In [11]:
# %%
print("=" * 60)
print("VALIDATION")
print("=" * 60)

occ_check = (
    df_out.groupby("occ_code")["normalized_time"]
    .sum(min_count=1)
    .reset_index(name="sum_normalized")
)

print("Normalized sums, sample:")
print(occ_check.head(10))

print("\nDistribution of normalized sums:")
print(occ_check["sum_normalized"].describe())

VALIDATION
Normalized sums, sample:
     occ_code  sum_normalized
0  11-1011.00             1.0
1  11-1011.03             1.0
2  11-1021.00             1.0
3  11-1031.00             1.0
4  11-2011.00             1.0
5  11-2021.00             1.0
6  11-2022.00             1.0
7  11-2032.00             1.0
8  11-2033.00             1.0
9  11-3012.00             1.0

Distribution of normalized sums:
count    9.230000e+02
mean     1.000000e+00
std      1.864367e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: sum_normalized, dtype: float64
